In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "notebooks") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

<div style="width: 100%; height: 5px; background-color: green;"></div>
Let's test the dataset to make sure we are getting what we expect.    <br>
We'll create the Confirguration file, and load a dataset - just as we will when we train.  <br>

The dataset is a HuggingFace dataset with 3 spits, train, train_sm, and val. <br>
Audio is saved as 8-tokens per frame, 75 frames per second using the 24kHz pretrained Encodec model on Hugging Face. <br>

The dataset itself decodes the audio to the 128D latent vectors that is the intermediate representation between audio and quantization for coded. <br>
Note that we use the *reconstructed* latents (decoded from code tokens),  rather than the latens encoded from audio. We want to limit input data to latent values that the model will see during inference when they are taken from the output of the previous step that predicts codes, not latents. <br>

We visualize a sample, most importantly the distribution of latent vector component values - because we are going to clip them (and then normalize them) to prepare them as input for training a model. The last part of this notebook resynthesizes the audio from the clipped latents as a check that our clipping threshold is reasonable (doesn't affect the audio reconstruction too much). 

<div style="width: 100%; height: 5px; background-color: green;"></div>

In [ ]:
# EnCodec Latent Dataset Testing Notebook

import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
#import seaborn as sns
from IPython.display import Audio, display
import pandas as pd
from pathlib import Path

# Import your dataset class (adjust import path as needed)
from rnencodec.audioDataLoader.audio_dataset import EnCodecLatentDataset, LatentDatasetConfig, efficient_codes_to_latents, EnCodecLatentDataset_dynamic_v2
from transformers import EncodecModel

# Set up plotting style
plt.style.use('default')
#sns.set_palette("viridis")

<div style="width: 100%; height: 10px; background-color: green;"></div>
<b> parameters <b>

In [ ]:
# The syn7wav24_tokens_HFdataset_5 has the following parameters: 
#         ['Full File Name', 'class_name', 'param1', 'param2', 'param3', 'param0', 'audio']
# and the following unique 'class_name' values:
#         Class list: ['ChirpPattern', 'DSApplause', 'DSBugs', 'DSPeepers', 'DSPistons', 'DSWind', 'TokWotalDuet']
# which you can use to filter the full dataset if you only want to train on a subset of sound classes, for example:
#         config.filters = {'class_name': {'DSPistons', 'DSApplause'}}      

# there are 40425 sequences from 147 files in 'val' split - 7 classes, 21 param values, 274 starting frames possible in each file
sample_idx = 4*275  # You can change this or use random  (tok, wind, Peepers,  wind,  bugs, chirp, applause, pistons)
numcodebooks = 4

In [ ]:
config = LatentDatasetConfig(
    #dataset_path="/slowdisk/esteban/scratchdata/syntex24/data7wav/syn7wav24_tokens_HFdataset_5",  # Update this path
    dataset_path="/slowdisk/data/estaban_babble_gender/dataset_hf",
    sequence_length=75,  # Number of frames to test with
    parameter_specs={
        # "pitch": None, #(0, 1),  # Update with your actual parameter ranges
        # "amp": None  #(0, 1.0),  # Update with your actual parameter ranges
        "sex": None, #(0, 1),  # Update with your actual parameter ranges
        # Add more parameters as needed
    },
    n_q=numcodebooks,  # Number of quantization levels to use
    codebook_size=1024,
    add_noise=False,
    clamp_val = 15.0, # latents are clamped between [-clamp_val , clamp_val] and then mapped to [-1,1] for training
    filters= None, #{'class_name': {'DSPistons'}} 
    files_per_sequence = 1
)


# Load your EnCodec model (update path as needed)
# from transformers import EncodecModel
#We load a model to use in the notebook, but to create the dataset, we pass it a path to the model and it creates it's own
model = EncodecModel.from_pretrained("facebook/encodec_24khz")
model.eval()

print("✅ Configuration and model loaded")

In [ ]:
# Initialize the dataset

print("Loading specific dataset...")
dataset = EnCodecLatentDataset_dynamic_v2(config, "facebook/encodec_24khz", split='train')  # or 'train'
print(f"Dataset loaded with {len(dataset)} sequences")

In [ ]:
dataset

In [ ]:
# Peek columns (optional)
#print(dataset.dataset.column_names)
#print(ds.features)  # shows types per column

#print(f"Class list: {dataset.getUniqueStrings('class_name')}")


# Get a random sample
print(f'Sample index is {sample_idx}')

input_tensor, target_codes = dataset[sample_idx]

print(f"\n📊 Sample {sample_idx} Statistics:")
print(f"Input tensor shape: {input_tensor.shape}")  # Should be (seq_len, 128 + n_conditioning)
print(f"Target codes shape: {target_codes.shape}")  # Should be (seq_len, n_q)
print(f"Input tensor dtype: {input_tensor.dtype}")
print(f"Target codes dtype: {target_codes.dtype}")

# Extract latent vectors and conditioning
latent_vectors = input_tensor[:, :128]  # First 128 dimensions are latents
conditioning = input_tensor[:, 128:]    # Remaining dimensions are conditioning

print(f"Latent vectors shape: {latent_vectors.shape}")
print(f"Conditioning shape: {conditioning.shape}")
print(f"Conditioning values (first frame): {conditioning[0]}")

In [ ]:
dataset[sample_idx]

In [ ]:
# Get the original dataset record for context
print(f'Sample index is {sample_idx}')
dataset_idx, start_frame, token_file_path = dataset.sequence_map[sample_idx]
original_row = dataset.dataset[dataset_idx]

print("📋 Original Dataset Record:")
print("-" * 50)
for key, value in original_row.items():
    if isinstance(value, (int, float, str)):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {type(value)} (shape: {getattr(value, 'shape', 'N/A')})")

print(f"\n🎯 Sequence Info:")
print(f"Dataset index: {dataset_idx}")
print(f"Start frame: {start_frame}")
print(f"End frame: {start_frame + config.sequence_length}")
print(f"Token file path: {token_file_path}")

<div style="width: 100%; height: 10px; background-color: green;"></div>
<b> Show heatmap of latent seequence and the distribution of values <b>

In [ ]:
# Plot 1: Heatmap of 128D latent vectors over time
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Main heatmap
ax1 = axes[0, 0]
im = ax1.imshow(latent_vectors.T, aspect='auto', cmap='viridis', interpolation='nearest')
ax1.set_title('128D Latent Vector Sequence Heatmap')
ax1.set_xlabel('Time Steps')
ax1.set_ylabel('Latent Dimensions')
plt.colorbar(im, ax=ax1, label='Activation Value')

# Magnitude over time (average across dimensions)
ax2 = axes[0, 1]
magnitude_over_time = torch.abs(latent_vectors).mean(dim=1)
ax2.plot(magnitude_over_time.numpy())
ax2.set_title('Average Magnitude Over Time')
ax2.set_xlabel('Time Steps')
ax2.set_ylabel('Average |Activation|')
ax2.grid(True, alpha=0.3)

# Magnitude per dimension (average across time)
ax3 = axes[1, 0]
magnitude_per_dim = torch.abs(latent_vectors).mean(dim=0)
ax3.bar(range(128), magnitude_per_dim.numpy(), alpha=0.7)
ax3.set_title('Average Magnitude Per Dimension')
ax3.set_xlabel('Latent Dimension')
ax3.set_ylabel('Average |Activation|')
ax3.grid(True, alpha=0.3)

# Distribution of activation values
ax4 = axes[1, 1]
ax4.hist(latent_vectors.flatten().numpy(), bins=50, alpha=0.7, density=True)
ax4.set_title('Distribution of Latent Activations')
ax4.set_xlabel('Activation Value')
ax4.set_ylabel('Density')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📈 Latent Statistics:")
print(f"Min activation: {latent_vectors.min().item():.4f}")
print(f"Max activation: {latent_vectors.max().item():.4f}")
print(f"Mean activation: {latent_vectors.mean().item():.4f}")
print(f"Std activation: {latent_vectors.std().item():.4f}")

total_elements = latent_vectors.numel()  # Total number of elements|
epsilon=.0001
near_pos_one = (torch.abs(latent_vectors - 1) <= epsilon).sum().item()
near_neg_one = (torch.abs(latent_vectors + 1) <= epsilon).sum().item()
print(f"Clamped at +1: {near_pos_one} ({near_pos_one/total_elements*100:.2f}%)")
print(f"Clamped at -1: {near_neg_one} ({near_neg_one/total_elements*100:.2f}%)")
print(f"Total clamped ({(near_pos_one + near_neg_one)/total_elements*100:.2f}%)")

In [ ]:
# Plot 2: Target codes analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Heatmap of target codes
ax1 = axes[0, 0]
im = ax1.imshow(target_codes.T, aspect='auto', cmap='tab20', interpolation='nearest')
ax1.set_title(f'Target Codes Heatmap ({config.n_q} Quantizers)')
ax1.set_xlabel('Time Steps')
ax1.set_ylabel('Quantizer Level')
plt.colorbar(im, ax=ax1, label='Code Index')

# Distribution of codes per quantizer
for i in range(config.n_q):
    ax2 = axes[0, 1] if i < 2 else axes[1, i-2]
    codes_for_quantizer = target_codes[:, i]
    ax2.hist(codes_for_quantizer.numpy(), bins=50, alpha=0.7, 
             label=f'Quantizer {i}')
    ax2.set_title(f'Code Distribution - Quantizer {i}')
    ax2.set_xlabel('Code Index')
    ax2.set_ylabel('Frequency')
    ax2.grid(True, alpha=0.3)

# Overall code statistics
axes[1, 1].text(0.1, 0.8, f"Target Code Statistics:", transform=axes[1, 1].transAxes, fontsize=12, fontweight='bold')
stats_text = f"""
Min code: {target_codes.min().item()}
Max code: {target_codes.max().item()}
Unique codes total: {len(torch.unique(target_codes))}
Codes per quantizer: {[len(torch.unique(target_codes[:, i])) for i in range(config.n_q)]}
"""
axes[1, 1].text(0.1, 0.6, stats_text, transform=axes[1, 1].transAxes, fontsize=10, verticalalignment='top')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Reconstruct audio from the latent vectors
print("🔄 Reconstructing audio from latent vectors...")

# We need to convert latents back to audio via the decoder
with torch.no_grad():
    # Reshape latents for decoder: (seq_len, 128) -> (1, 128, seq_len)
    latents_for_decoder = latent_vectors.T.unsqueeze(0)
    
    print(f"Latents shape for decoder: {latents_for_decoder.shape}")
    
    # Pass through decoder to get audio
    reconstructed_audio = model.decoder(latents_for_decoder*config.clamp_val)   #un normalize them for reconstruction!
    
    # Remove batch dimension and convert to numpy
    audio_np = reconstructed_audio.squeeze(0).squeeze(0).cpu().numpy()
    
    print(f"Reconstructed audio shape: {audio_np.shape}")
    print(f"Audio duration: {len(audio_np) / 24000:.2f} seconds")

# Plot audio waveform
plt.figure(figsize=(15, 6))
time_axis = np.arange(len(audio_np)) / 24000  # 24kHz sampling rate

plt.subplot(2, 1, 1)
plt.plot(time_axis, audio_np, alpha=0.8)
plt.title('Reconstructed Audio Waveform')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)

# Plot spectrogram
plt.subplot(2, 1, 2)
from scipy import signal
f, t, Sxx = signal.spectrogram(audio_np, fs=24000, nperseg=512)
plt.pcolormesh(t, f, 10 * np.log10(Sxx), shading='gouraud', cmap='viridis')
plt.title('Reconstructed Audio Spectrogram')
plt.xlabel('Time (seconds)')
plt.ylabel('Frequency (Hz)')
plt.ylim(0, 8000)  # Focus on up to 8kHz
plt.colorbar(label='Power (dB)')

plt.tight_layout()
plt.show()

In [ ]:
# Create audio player
print("🎵 Audio Player:")
print(f"Sample rate: 24000 Hz")
print(f"Duration: {len(audio_np) / 24000:.2f} seconds")

# Normalize audio for playback
audio_normalized = audio_np / np.max(np.abs(audio_np)) * 0.8  # Prevent clipping

# Display audio player
display(Audio(audio_normalized, rate=24000))

In [ ]:
# Additional Analysis: Conditioning parameters over time
if conditioning.shape[1] > 0:
    plt.figure(figsize=(12, 6))
    
    param_names = list(config.parameter_specs.keys())
    n_params = conditioning.shape[1]
    
    for i in range(n_params):
        plt.subplot(2, (n_params + 1) // 2, i + 1)
        plt.plot(conditioning[:, i].numpy())
        param_name = param_names[i] if i < len(param_names) else f'param_{i}'
        plt.title(f'Conditioning: {param_name}')
        plt.xlabel('Time Steps')
        plt.ylabel('Normalized Value [0,1]')
        plt.grid(True, alpha=0.3)
        plt.ylim(-0.1, 1.1)
    
    plt.tight_layout()
    plt.show()
    
    print("📊 Conditioning Parameters:")
    for i, param_name in enumerate(param_names[:n_params]):
        value = conditioning[0, i].item()  # Same across time
        print(f"{param_name}: {value:.4f}")

In [ ]:
# Performance and Memory Analysis
import psutil
import time

print("⚡ Performance Analysis:")
print("-" * 50)

# Memory usage
process = psutil.Process()
memory_mb = process.memory_info().rss / 1024 / 1024
print(f"Current memory usage: {memory_mb:.1f} MB")

# Dataset loading speed test
print("\n⏱️  Speed Test (loading 10 random samples):")
start_time = time.time()

for _ in range(10):
    idx = np.random.randint(0, len(dataset))
    sample_input, sample_target = dataset[idx]

end_time = time.time()
avg_time_per_sample = (end_time - start_time) / 10

print(f"Average time per sample: {avg_time_per_sample:.4f} seconds")
print(f"Estimated samples per second: {1/avg_time_per_sample:.1f}")

# DataLoader speed test =======================================================================================================
from torch.utils.data import DataLoader

print("\n📦 DataLoader Speed Test:")
dataloader = DataLoader(dataset, batch_size=24, shuffle=True, num_workers=0)

start_time = time.time()
batch_count = 0
for batch_input, batch_target in dataloader:
    batch_count += 1
    if batch_count >= 5:  # Test 5 batches
        break

end_time = time.time()
total_samples = batch_count * 4  # batch_size = 4
avg_time_per_batch = (end_time - start_time) / batch_count

print(f"Batches processed: {batch_count}")
print(f"Average time per batch: {avg_time_per_batch:.4f} seconds")
print(f"Samples per second (batched): {total_samples/(end_time - start_time):.1f}")

In [ ]:
# Summary Report
print("📋 DATASET TEST SUMMARY")
print("=" * 60)
print(f"✅ Dataset successfully loaded: {len(dataset)} sequences")
print(f"✅ Sample successfully extracted and processed")
print(f"✅ 128D latent vectors generated from codes")
print(f"✅ Audio reconstruction working")
print(f"✅ Conditioning parameters properly normalized")
print(f"✅ Target codes properly formatted for training")

print(f"\n📊 Key Metrics:")
print(f"   - Input tensor shape: {input_tensor.shape}")
print(f"   - Target codes shape: {target_codes.shape}")
print(f"   - Latent magnitude range: [{latent_vectors.min().item():.3f}, {latent_vectors.max().item():.3f}]")
print(f"   - Unique target codes: {len(torch.unique(target_codes))}")
print(f"   - Audio reconstruction length: {len(audio_np)} samples ({len(audio_np)/24000:.2f}s)")

print(f"\n🚀 Ready for training!")

<div style="width: 100%; height: 10px; background-color: green;"></div>
<b> Now explore audio quality for clipped latents <b>

In [ ]:
import torch

@torch.no_grad()
def latents_to_audio_simple(model, embeddings):
    """
    EnCodec 24kHz mono: 128-D latents -> waveform.
    No normalization, no PQMF.

    Args:
        model: EnCodec model (24 kHz, mono).
        embeddings: (B, T_frames, 128) or (B, 128, T_frames)

    Returns:
        wave: (B, 1, T_audio) float32 on CPU
        sr:   int (should be 24000 for your model)
    """
    z = embeddings
    if z.dim() != 3:
        raise ValueError(f"Expected (B, T, 128) or (B, 128, T), got {tuple(z.shape)}")

    # Ensure (B, 128, T)
    if z.shape[1] != 128 and z.shape[2] == 128:
        z = z.transpose(1, 2)

    # Decode latents to waveform
    x = model.decoder(z)              # usually returns a tensor directly

    # Unwrap if some variant returns (x, ...)
    if isinstance(x, (tuple, list)):
        x = x[0]
    elif isinstance(x, dict):
        x = x.get("x", next(iter(x.values())))

    # Move to CPU float32 for saving/playback
    x = x.detach().to(torch.float32).cpu()

    sr = int(getattr(model, "sample_rate", 24000))
    return x, sr


<div style="width: 100%; height: 10px; background-color: green;"></div>
<b> CLAMP  - set min and max to clip Latent Activation values before reconstructing audio! (see distribution of Latent Values visualization above) <b>

In [ ]:
# Latents are already clamped in [-config.clamp_val, config.clamp_val], but you can test tighter ranges
CLAMPMIN=-10
CLAMPMAX=10

clamped_latents = torch.clamp(latents_for_decoder*config.clamp_val, min=CLAMPMIN, max=CLAMPMAX) 
wave1, sr = latents_to_audio_simple(model, clamped_latents) 
wave1=wave1[0, 0]

wave=model.decoder(clamped_latents).detach().numpy()   #un normalize them for reconstruction!
wave=wave[0, 0]

(wave1-wave).max()

In [ ]:
# Plot audio waveform
plt.figure(figsize=(15, 6))
time_axis = np.arange(len(wave)) / 24000  # 24kHz sampling rate

plt.subplot(2, 1, 1)
plt.plot(time_axis, wave, alpha=0.8)
plt.title('Reconstructed Audio Waveform')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)

In [ ]:
display(Audio(wave, rate=int(sr)))

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b> CONTEXT ?????   grab middle segment of latents to reconstruct with less context! <b>

In [ ]:
print(f'Shape of latents_for_decode is {latents_for_decoder.shape}')
middle_latents=latents_for_decoder[:,:,33:-33]
print(f'Shape of middle_latents is {middle_latents.shape}')
clamped_latents = torch.clamp(middle_latents*config.clamp_val, min=CLAMPMIN, max=CLAMPMAX) 
middle_wave, sr = latents_to_audio_simple(model, clamped_latents) 
middle_wave=middle_wave[0, 0]


In [ ]:
# Plot audio waveform
plt.figure(figsize=(15, 6))
time_axis = np.arange(len(middle_wave)) / 24000  # 24kHz sampling rate

plt.subplot(2, 1, 1)
plt.plot(time_axis, middle_wave, alpha=0.8)
plt.title('Reconstructed Audio Waveform')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)

In [ ]:
display(Audio(middle_wave, rate=int(sr)))